In [1]:
import pandas as pd



In [2]:
# Carregando os arquivos CSV em DataFrames
df1 = pd.read_csv("Divvy_Trips_2019_Q1.csv")
df2 = pd.read_csv("Divvy_Trips_2020_Q1.csv")

In [3]:
# Verificando o número de linhas e colunas em cada DataFrame
print(df1.shape)
print(df2.shape)


(365069, 12)
(426887, 13)


In [4]:
# Verificando os nomes das colunas em cada DataFrame
print("\nColunas df1:")
print(df1.columns)

print("\nColunas df2:")
print(df2.columns)


Colunas df1:
Index(['trip_id', 'start_time', 'end_time', 'bikeid', 'tripduration',
       'from_station_id', 'from_station_name', 'to_station_id',
       'to_station_name', 'usertype', 'gender', 'birthyear'],
      dtype='str')

Colunas df2:
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')


In [5]:
# Padronizando os nomes das colunas do df1 para corresponder ao df2
df1 = df1.rename(columns={
    'trip_id': 'ride_id',
    'start_time': 'started_at',
    'end_time': 'ended_at',
    'from_station_name': 'start_station_name',
    'to_station_name': 'end_station_name',
    'usertype': 'member_type'
})

In [6]:
# Selecionando apenas as colunas necessárias em df1
df1 = df1[
    [
        'ride_id',
        'started_at',
        'ended_at',
        'start_station_name',
        'end_station_name',
        'member_type'
    ]
]


In [7]:
# Padronizando os nomes das colunas do df2 para corresponder ao df1
df2 = df2.rename(columns={
    'member_casual': 'member_type'
})

In [8]:
# Selecionando apenas as colunas necessárias em df2
df2 = df2[
    [
        'ride_id',
        'started_at',
        'ended_at',
        'start_station_name',
        'end_station_name',
        'member_type'
    ]
]


In [9]:
# Concatenando os DataFrames df1 e df2
df = pd.concat([df1, df2])
df = df.reset_index(drop=True)


In [10]:
# Verificando o número de linhas e colunas no DataFrame concatenado
df.shape
df.head()

,ride_id,started_at,ended_at,start_station_name,end_station_name,member_type
0,21742443,2019-01-01 00:04:37,2019-01-01 00:11:07,Wabash Ave & Grand Ave,Milwaukee Ave & Grand Ave,Subscriber
1,21742444,2019-01-01 00:08:13,2019-01-01 00:15:34,State St & Randolph St,Dearborn St & Van Buren St (*),Subscriber
2,21742445,2019-01-01 00:13:23,2019-01-01 00:27:12,Racine Ave & 18th St,Western Ave & Fillmore St (*),Subscriber
3,21742446,2019-01-01 00:13:45,2019-01-01 00:43:28,California Ave & Milwaukee Ave,Clark St & Elm St,Subscriber
4,21742447,2019-01-01 00:14:52,2019-01-01 00:20:56,Mies van der Rohe Way & Chicago Ave,Streeter Dr & Grand Ave,Subscriber


In [11]:
# Verificando os tipos de dados das colunas no DataFrame concatenado
df.dtypes

ride_id               object
started_at               str
ended_at                 str
start_station_name       str
end_station_name         str
member_type              str
dtype: object

In [12]:
# Convertendo as colunas 'started_at' e 'ended_at' para o tipo datetime
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])


In [13]:
# Criando a coluna 'ride_length'
df['ride_length'] = df['ended_at'] - df['started_at']


In [14]:
# Criando a coluna 'ride_length_min' convertendo a duração da viagem para minutos
df['ride_length_min'] = df['ride_length'].dt.total_seconds() / 60


In [15]:
# Verificando se há durações inválidas
df['ride_length'].describe()

count                    791956
mean     0 days 00:19:43.813276
std      0 days 09:13:32.741229
min           -1 days +23:50:48
25%             0 days 00:05:28
50%             0 days 00:08:57
75%             0 days 00:15:10
max           123 days 01:20:22
Name: ride_length, dtype: object

In [16]:
# Verificando quantas viagens são inválidas (com duração menor ou igual a zero)
(df['ride_length'] <= pd.Timedelta(0)).sum()

np.int64(210)

In [17]:
# Removendo as viagens inválidas (com duração menor ou igual a zero)
df = df[df['ride_length'] > pd.Timedelta(0)]

In [18]:
# Recomendo viagens com duração superior a 12 horas por representarem outliers extremos, potencialmente relacionados a falhas de devolução ou uso não típico do serviço.
df = df[df['ride_length'] <= pd.Timedelta(hours=12)]

In [19]:
# Removendo viagens com duração inferior a 1 minuto por não representarem uso significativo do serviço, podem ser erros ou desistências imediatas
df = df[df['ride_length_min'] >= 1]



In [20]:
# Verificando as estatísticas após a limpeza dos dados
df['ride_length'].describe()

count                    783388
mean     0 days 00:13:14.298771
std      0 days 00:19:25.847977
min             0 days 00:01:00
25%             0 days 00:05:33
50%             0 days 00:09:01
75%             0 days 00:15:13
max             0 days 11:56:42
Name: ride_length, dtype: object

In [21]:
# Criando a coluna 'day_of_week' para análise de padrões de uso por dia da semana
df['day_of_week'] = df['started_at'].dt.day_name()


In [22]:
# Ordenando os dias da semana para melhor visualização
order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday', 'Saturday', 'Sunday'
]

df['day_of_week'] = pd.Categorical(df['day_of_week'], categories=order, ordered=True)
df['day_of_week'].value_counts().sort_index()


day_of_week
Monday       115367
Tuesday      134556
Wednesday    128875
Thursday     131601
Friday       122473
Saturday      72369
Sunday        78147
Name: count, dtype: int64

In [23]:
# Criando a coluna 'month' para análise de padrões de uso por mês
df['month'] = df['started_at'].dt.month_name()

In [24]:
# Ordenando os meses para melhor visualização
month_order = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)
df['month'].value_counts().sort_index()
# A análise mensal é limitada aos meses de janeiro, fevereiro e março, pois os conjuntos de dados utilizados correspondem apenas ao primeiro trimestre (Q1) de 2019 e 2020.

month
January      245633
February     233630
March        304125
April             0
May               0
June              0
July              0
August            0
September         0
October           0
November          0
December          0
Name: count, dtype: int64

In [25]:
# Verificando a distribuição dos tipos de membros
df['member_type'].value_counts()


member_type
member        374532
Subscriber    341685
casual         44122
Customer       23049
Name: count, dtype: int64

In [26]:
# Padronizando os valores da coluna 'member_type' para facilitar a análise
df['member_type'] = df['member_type'].replace({
    'Subscriber': 'member',
    'Customer': 'casual'
})


In [27]:
# Duração média das viagens por tipo de membro
df.groupby('member_type')['ride_length_min'].mean()


member_type
casual    35.055139
member    11.192204
Name: ride_length_min, dtype: float64

In [28]:
# Analisando a duração média das viagens por tipo de membro
df.groupby('member_type')['ride_length_min'].mean().round(0)


member_type
casual    35.0
member    11.0
Name: ride_length_min, dtype: float64

In [29]:
# Analisando a distribuição de viagens por dia da semana e tipo de membro
df.groupby(['day_of_week', 'member_type']).size()


day_of_week  member_type
Monday       casual           5540
             member         109827
Tuesday      casual           7236
             member         127320
Wednesday    casual           7600
             member         121275
Thursday     casual           7060
             member         124541
Friday       casual           7918
             member         114555
Saturday     casual          13357
             member          59012
Sunday       casual          18460
             member          59687
dtype: int64

In [30]:
# Conferindo o DataFrame final após as transformações e limpezas
df.info()

<class 'pandas.DataFrame'>
Index: 783388 entries, 0 to 791955
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype          
---  ------              --------------   -----          
 0   ride_id             783388 non-null  object         
 1   started_at          783388 non-null  datetime64[us] 
 2   ended_at            783388 non-null  datetime64[us] 
 3   start_station_name  783388 non-null  str            
 4   end_station_name    783388 non-null  str            
 5   member_type         783388 non-null  str            
 6   ride_length         783388 non-null  timedelta64[us]
 7   ride_length_min     783388 non-null  float64        
 8   day_of_week         783388 non-null  category       
 9   month               783388 non-null  category       
dtypes: category(2), datetime64[us](2), float64(1), object(1), str(3), timedelta64[us](1)
memory usage: 55.3+ MB


In [31]:
# Selecionando apenas as colunas relevantes para análise de BI
df_bi = df[
    [
        'ride_id',
        'started_at',
        'member_type',
        'ride_length_min',
        'day_of_week',
        'month'
    ]
]
# Exportando o DataFrame limpo e preparado para análise de BI
df_bi.to_csv(
    "divvy_trips_q1_2019_2020_cleaned_br.csv",
    index=False,
    sep=";",
    decimal=","
)